In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import metrics, tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,                              
                             f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)



In [ ]:
random_variable = 1234

Path to files on server 
- example Mac file_path = "~/name/kurse/ppa/session_2/"
- example Windows file_path = "c:/users/name/kurse/session_2/"

In [ ]:
file_path = "https://raw.github.com/jan-source/ppa_erp_student/main/sessions/2_session/"

### Read data

In [ ]:
data_a = pd.read_csv(f"{file_path}churn_part_a.csv")

In [ ]:
data_a.head()

In [ ]:
data_a.dtypes

In [ ]:
data_b = pd.read_csv(f"{file_path}churn_part_b.csv")

In [ ]:
data_b.head()

In [ ]:
data_b.dtypes

### combine both datasets (keep only rows which exists in data_a and data_b)

In [ ]:
data = pd.merge(left=data_a, right=data_b, on="CustomerId", how="inner")

In [ ]:
data.head()

### Which % of customers churned

In [ ]:
data["Exited"].mean()

### How many rows have an NaN in any column

simulate how many rows are left when deleting rows with one empty column

In [ ]:
data.dropna(axis=0).shape

In [ ]:
data.shape

delete rows with one empty column

In [ ]:
data = data.dropna(axis=0)

In [ ]:
data.shape

delete only Rows if missing values in specific columns 
data.dropna(axis=0, subset=["Gender", "Age"])

In [ ]:
plt.scatter(data['Age'], data['Exited'])

In [ ]:
# Create age groups and analyze churn by age with counts
data.loc[:, 'Age_Group'] = pd.cut(
        data['Age'], 
        bins=[17, 30, 40, 50, 60, 100], 
        labels=['18-30', '31-40', '41-50', '51-60', '60+']
        )

# Calculate churn rate and counts for each age group
age_group_stats = data.groupby('Age_Group').agg(
    churn_rate=('Exited', 'mean'),
    count=('Exited', 'count')
)

# Create the bar chart
plt.figure(figsize=(10, 6))
age_group_stats['churn_rate'].plot(kind='bar', color='blue', alpha=0.7)
plt.title('Customer Churn Rate by Age Group', fontsize=14)
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Churn Rate', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)


plt.ylim(0, 0.7)  # Sets y-axis to range from 0 to 0.5 (or 50%)

# Add percentage labels and counts on top of each bar
for i, (rate, count) in enumerate(zip(age_group_stats['churn_rate'], age_group_stats['count'])):
    plt.text(i, rate + 0.01, f'{rate:.1%}\n(n={count})', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## Logistic Regression

Define Features

In [ ]:
x = data[["Age", "CreditScore", "Balance"]]

Define Target

In [ ]:
y = data["Exited"]

Init Model

In [ ]:
log_reg_simple = LogisticRegression(random_state=random_variable)

Train Model

In [ ]:
log_reg_simple.fit(x, y)

Ausgabe Koeffizienten:

In [ ]:
log_reg_simple.coef_

In [ ]:
coefficients_formated =  pd.concat(
    [
     pd.Series(['Age', 'CreditScore', 'Balance']),
     pd.DataFrame(np.transpose(log_reg_simple.coef_))
    ],
     axis=1
 )

In [ ]:
coefficients_formated

Calculate Predictions

Predictions 1 / 0

In [ ]:
predictions = log_reg_simple.predict(x)

In [ ]:
len(log_reg_simple.predict(x))

For how many customers does the model predict 1 (customer leaves us)

In [ ]:
sum(log_reg_simple.predict(x))

Get the probabibility for each customer that she/he leaves us 
- first value is probability for 0 (customer stays)
- second value is probability for 1 (customer leaves)

In [ ]:
log_reg_simple.predict_proba(x)

## Calculate Confusion Matrix (as in Slides 39)

In [ ]:
confusion_matrix(y, predictions, labels=[True, False]).T

Calculate accuracy by hand

In [ ]:
(125 + 7674) / (125 + 286 + 1911 + 7674) 

Use implemented sklearn function

In [ ]:
accuracy_score(y, predictions)

Calculate Precision by hand

In [ ]:
125 / (125 + 286)

Calculate Recall by hand

In [ ]:
125 / (125 + 1911)